# NeuroPersona — train the OCEAN autoencoder

Trains the model that assigns a personality type, then exports it as plain numpy
matrices for the Flask backend to serve.

**Dataset** Open-Source Psychometrics Project, *Big Five Personality Test*:
1,015,342 responses to the 50-item IPIP Big-Five Factor Markers, public domain.
Add it to this notebook with **+ Add Input → Datasets → search `big-five-personality-test`**
(owner `tunguz`).

**Settings** Accelerator: GPU T4 or P100 (CPU also works, just slower).
Internet: **On**, so the cell below can fetch the training module.

**Runtime** A full 30-epoch run over ~850k cleaned rows takes roughly 15-25 minutes
on a T4. The checkpoint/resume path below exists so a much longer run can span
several sessions across a week without losing progress, since one Kaggle session
is capped at 12 hours.


## 1. Get the training module

One source of truth: the module lives in the repository, so the notebook cannot
drift away from the code the backend was built against. If Internet is off, upload
`training/train_vae_ocean.py` as a Kaggle Dataset instead and point `sys.path` at it.


In [ ]:
import os, sys, subprocess

REPO = "https://github.com/udj171/NeuroPersona.git"

if not os.path.isdir("/kaggle/working/NeuroPersona"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/NeuroPersona"], check=True)

sys.path.insert(0, "/kaggle/working/NeuroPersona/training")
import train_vae_ocean as T
print("items:", len(T.ITEM_ORDER), "| traits:", T.TRAIT_ORDER, "| latent:", T.LATENT_DIM)


## 2. Point at the data

The Kaggle dataset unpacks to a tab-separated file. Adjust the path if the dataset
version you attached lays it out differently.


In [ ]:
DATA = "/kaggle/input/big-five-personality-test/IPIP-FFM-data-8Nov2018/data-final.csv"
OUT  = "/kaggle/working"

assert os.path.exists(DATA), (
    f"Not found: {DATA}\n"
    "Add the dataset with + Add Input, then check the path under /kaggle/input."
)
print(f"{os.path.getsize(DATA) / 1e6:.0f} MB")


## 3. Smoke run first

Two epochs over 50,000 rows. This proves the columns parse, the shapes line up and
the export writes, in about a minute. Do not skip it before a long run.


In [ ]:
_ = T.train(data_path=DATA, out_dir="/kaggle/working/smoke", epochs=2, max_rows=50_000)


## 4. Full run

Every epoch writes `checkpoint.pt` to `OUT`. To continue in a later session:

1. Save this notebook version so `/kaggle/working` is kept as output.
2. **+ Add Input → Notebook Output**, attach that output to the new session.
3. Pass its path as `resume_dir` below.

Training resumes at the next epoch rather than starting over.


In [ ]:
RESUME_DIR = None   # e.g. "/kaggle/input/neuropersona-vae-run-1"

weights_path = T.train(
    data_path=DATA,
    out_dir=OUT,
    epochs=30,
    resume_dir=RESUME_DIR,
)
print(weights_path)


## 5. Check the export before trusting it

Loads the file back with numpy only, exactly as the backend will, and runs one
profile through it. If this cell is happy, `model_inference.py` will be too.


In [ ]:
import numpy as np

w = np.load(weights_path, allow_pickle=False)
print("keys:", sorted(w.files))
assert [str(x) for x in w["item_order"]] == T.ITEM_ORDER, "item order drifted"

x = (np.full(50, 3.0) - w["feature_mean"]) / (w["feature_std"] + 1e-8)
h = np.tanh(x @ w["enc_w1"] + w["enc_b1"])
h = np.tanh(h @ w["enc_w2"] + w["enc_b2"])
z = h @ w["enc_mu_w"] + w["enc_mu_b"]
d = np.linalg.norm(w["centroids"] - z, axis=1)

print("latent:", z.round(3))
print("nearest type:", str(w["type_names"][int(d.argmin())]))
print("types:", [str(n) for n in w["type_names"]])
print("meta:", str(w["meta_json"]))


## 6. Install the weights

Download `ocean_vae.npz` from the notebook output and commit it to the repository at
`backend/model_weights/ocean_vae.npz`, or set `MODEL_WEIGHTS_PATH` to wherever you
put it. The backend loads it at startup and logs whether it succeeded.

The file is a few hundred kilobytes: small enough to commit, which keeps deployment
to a single push.

Until the file is installed the app still runs. It scores every profile and writes an
interpretation, and says plainly that no trained model is behind the type.


In [ ]:
print(f"{os.path.getsize(weights_path) / 1e3:.0f} KB  ->  backend/model_weights/ocean_vae.npz")
